In [ ]:
# ========== 客服回复副驾驶 v2：总览（理念） ==========
# Support Reply Copilot：对比云端/本地回答 + Judge 打分 + SQLite 宏库 + 可选 TTS
#
# 你会练到的能力：
# - 双模型流式输出（Cloud vs Ollama）
# - Judge 模型输出严格 JSON 并挑赢家
# - SQLite 存放「已批准」客服宏（macros），降低胡编政策/SLA
# - DB 模式：Always / Auto（远程 tool-calling 决定）/ Off
# - 可选 TTS 朗读赢家回复；Gradio 把流程串成 UI
#
# 怎么跑：配置 OPENAI_API_KEY；本地需 `ollama serve` 且已 pull 对应模型；从上到下运行后 `demo.launch()`


In [ ]:
# ========== 1) 导入：环境、JSON、SQLite、Gradio、OpenAI ==========

# os：读环境变量与路径
import os
# json：解析 Judge 的 JSON、序列化 tool 返回
import json
# sqlite3：本地宏库（无需单独数据库服务）
import sqlite3
# 类型标注：Dict / Any / List，方便读接口
from typing import Dict, Any, List

# Gradio：快速搭对比 UI
import gradio as gr
# load_dotenv：从 .env 注入 OPENAI_API_KEY 等
from dotenv import load_dotenv
# OpenAI：云端官方端点 + Ollama 的 OpenAI 兼容端点都用这个客户端类
from openai import OpenAI


In [ ]:
# ========== 2) 环境 + 双客户端 + 默认模型名 ==========

# 加载 .env（override=True：覆盖进程里已有同名变量）
load_dotenv(override=True)

# 云端客户端：默认读环境变量 OPENAI_API_KEY
client_cloud = OpenAI()

# 本地客户端：指向 Ollama 的 OpenAI 兼容 /v1；api_key 占位即可
# 前提：本机已 `ollama serve`
client_local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 默认模型名（字符串须与账号/本机已装模型一致）
MODEL_1_DEFAULT = "gpt-4.1-nano"   # Cloud / Model 1
MODEL_2_DEFAULT = "llama3.1:8b"    # Local / Model 2 (Ollama)
JUDGE_DEFAULT   = "gpt-4.1-mini"   # Judge（云端）

# 固定温度：略随机但尽量稳，UI 不再暴露该旋钮
TEMPERATURE_FIXED = 0.2  # Keep small randomness but stable

# 探测 Ollama 是否可达：list models 当 ping
try:
    _ = client_local.models.list()  # Simple ping to local server
    ollama_ok = True
except Exception:
    ollama_ok = False


In [ ]:
# ========== 3) SQLite：建表 + 播种已批准客服宏 ==========

# DB 文件名：落在笔记本同目录
DB_PATH = "support_macros.db"

def init_macros_db(db_path: str) -> None:
    """创建 macros 表；若为空则写入一小批已批准客服宏模板。"""
    # connect：文件不存在则创建
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()  # Cursor 用来执行 SQL

        # IF NOT EXISTS：重复运行笔记本也不会报错
        cur.execute("""
        CREATE TABLE IF NOT EXISTS macros (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            intent TEXT NOT NULL,
            title TEXT NOT NULL,
            content TEXT NOT NULL,
            tags TEXT NOT NULL
        );
        """)

        # 数一下已有行，避免重复 seed
        cur.execute("SELECT COUNT(*) FROM macros;")
        count = cur.fetchone()[0]

        # 仅空表时插入种子数据（宏正文保持英文：会进 LLM 上下文）
        if count == 0:
            seed_rows = [
                # -------------------------
                # Billing / refunds
                # -------------------------
                (
                    "refund",
                    "Double charge / duplicate payment",
                    "Thanks for reporting this. I can see how frustrating that is. "
                    "Please share the invoice IDs (or the last 4 digits of the card + the charge dates), and we’ll verify the duplicate charge and process a refund if confirmed. "
                    "Once validated, refunds typically appear within 5–10 business days depending on your bank.",
                    "billing,refund,double charge,invoice,card"
                ),
                (
                    "billing_issue",
                    "Invoice / billing discrepancy",
                    "Thanks for reaching out. Please share your account email and the invoice number(s) affected, and tell us what looks incorrect (amount, plan, dates, taxes). "
                    "We’ll review and get back with a correction or explanation.",
                    "billing,invoice,pricing,taxes"
                ),

                # -------------------------
                # Login / authentication
                # -------------------------
                (
                    "login_help",
                    "Login issue after password reset",
                    "Sorry you’re having trouble logging in. Please confirm the email on the account and whether you see an error message. "
                    "If you recently reset your password, try clearing cache/cookies or using an incognito window; also confirm your device time is correct. "
                    "If it still fails, we can help verify the account and restore access.",
                    "login,password reset,auth,cache,cookies"
                ),
                (
                    "2fa_issue",
                    "2FA codes not arriving",
                    "Thanks for the details. If 2FA codes aren’t arriving, please check your spam folder and confirm the mailbox isn’t blocking automated emails. "
                    "If you use an authenticator app, confirm the app is synced to the correct time. "
                    "If you’re locked out, we can initiate a secure recovery—please share your account email and any recent successful login date you remember.",
                    "2FA,authentication,codes,email,authenticator"
                ),

                # -------------------------
                # Incident / outage
                # -------------------------
                (
                    "technical_outage",
                    "Service outage acknowledgement",
                    "Thanks for flagging this. We’re currently investigating the disruption and we’ll share updates as soon as we have confirmed details. "
                    "If you can, please send the approximate time it started, your region, and any error message or screenshot—this helps us correlate logs faster.",
                    "outage,incident,errors,region,screenshot"
                ),
                (
                    "eta_request",
                    "ETA request during incident",
                    "I understand you need an ETA. We’re actively working on the issue and will provide the next update by <TIME WINDOW>. "
                    "If you share your region and any error code you’re seeing, I can also confirm whether it matches the incident scope.",
                    "ETA,incident,update,scope"
                ),

                # -------------------------
                # Shipping / delivery (generic e-commerce)
                # -------------------------
                (
                    "shipping_delay",
                    "Shipping delay / late delivery",
                    "Sorry about the delay. Please share your order number and the delivery address postcode/ZIP, and I’ll check the latest carrier scan and expected delivery date. "
                    "If the package is stalled, we can start a carrier investigation.",
                    "shipping,delay,delivery,carrier,order"
                ),

                # -------------------------
                # Escalation / handoff
                # -------------------------
                (
                    "escalation",
                    "Escalate to specialist",
                    "Thanks—this looks like it needs a specialist. I’m escalating it now. "
                    "To speed things up, please include: your account email, exact steps to reproduce, time of occurrence, and any screenshots/logs. "
                    "We’ll follow up as soon as we have an update.",
                    "escalation,specialist,logs,screenshots"
                ),

                # -------------------------
                # Polite closing
                # -------------------------
                (
                    "closing",
                    "Polite closing",
                    "If you reply with the requested details, we’ll take it from there. Thanks for your patience.",
                    "closing,thanks,patience"
                ),
            ]

            # executemany：批量插入，占位符防 SQL 注入习惯
            cur.executemany(
                "INSERT INTO macros (intent, title, content, tags) VALUES (?, ?, ?, ?);",
                seed_rows
            )

        # commit：把建表/插入持久化到磁盘
        conn.commit()

# 初始化：创建或播种 DB
init_macros_db(DB_PATH)


In [ ]:
# ========== 4) search_macros：简单关键词 LIKE 检索 ==========

def search_macros(query: str, top_k: int = 3) -> Dict[str, Any]:
    """从顾客原话抽 token，用 LIKE 在 intent/title/content/tags 上 OR 匹配，返回 top_k 条。"""
    # 归一化：去空白、小写
    q = (query or "").strip().lower()
    if not q:
        return {"query": query, "hits": []}

    # 1) 极简分词：连续字母数字成 token，其它字符当分隔
    tokens = []
    current = []
    for ch in q:
        if ch.isalnum():
            current.append(ch)
        else:
            if current:
                tokens.append("".join(current))
                current = []
    if current:
        tokens.append("".join(current))

    # 2) 丢掉太短的噪声 token，并封顶 10 个以控制 SQL 大小
    tokens = [t for t in tokens if len(t) >= 3]
    tokens = tokens[:10]  # limit to keep query small and fast

    if not tokens:
        return {"query": query, "hits": []}

    # 3) 每个 token 对四个字段各 LIKE 一次，组间 OR
    where_clauses = []
    params = []
    for t in tokens:
        like = f"%{t}%"
        where_clauses.append("(lower(intent) LIKE ? OR lower(title) LIKE ? OR lower(content) LIKE ? OR lower(tags) LIKE ?)")
        params.extend([like, like, like, like])

    where_sql = " OR ".join(where_clauses)

    sql = f"""
        SELECT id, intent, title, content
        FROM macros
        WHERE {where_sql}
        LIMIT ?;
    """

    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute(sql, (*params, top_k))
        rows = cur.fetchall()

    # 组装 hits：excerpt 用于 UI 预览，content 全文留给 LLM
    hits: List[Dict[str, Any]] = []
    for mid, intent, title, content in rows:
        excerpt = content.strip()
        if len(excerpt) > 280:
            excerpt = excerpt[:280] + "..."
        hits.append({
            "id": mid,
            "intent": intent,
            "title": title,
            "excerpt": excerpt,
            "content": content
        })

    return {"query": query, "hits": hits}


In [ ]:
# ========== 5) format_db_trace：把检索结果收成 UI 用 Markdown ==========

def format_db_trace(db_result: Dict[str, Any]) -> str:
    """把 DB hits 收成一小段 Markdown，方便在 Gradio 里透明展示。"""
    # 无结果对象
    if not db_result:
        return "No DB lookup."

    # 取出 hits 列表
    hits = db_result.get("hits", [])

    # 明确告诉用户「查了但没命中」（英文 UI 文案保持原样）
    if not hits:
        return (
            "### DB macros\n"
            f"- Query: `{db_result.get('query','')}`\n"
            "- Result: **No hits**"
        )

    # 有命中：列出 id / intent / title
    lines = [
        "### DB macros",
        f"- Query: `{db_result.get('query','')}`",
        f"- Hits: **{len(hits)}**"
    ]
    for h in hits:
        lines.append(f"  - (id={h['id']}) **{h['intent']}** — {h['title']}")
    return "\n".join(lines)


In [ ]:
# ========== 6) 客服 Agent 的默认 system prompt（发给模型，保留英文） ==========

DEFAULT_SYSTEM_PROMPT = """
You are a professional customer support agent.
Your priority is factual accuracy and clarity.
You must not invent policies, SLAs, refunds, ETAs, pricing, or account details.
If information is missing, ask for the minimum necessary details.
Write a single email-style reply, concise and courteous, with clear next steps.
If an internal "APPROVED MACROS" reference is provided, reuse it and stay consistent with it.
Respond in English.
""".strip()


In [ ]:
# ========== 7) build_approved_macros_block：把 hits 变成高优先级 SYSTEM 参考 ==========

def build_approved_macros_block(db_result: Dict[str, Any]) -> str:
    """把 DB hits 转成 APPROVED MACROS 文本块；无命中也给「勿编造政策」护栏。"""
    # 取出 hits
    hits = db_result.get("hits", []) if db_result else []

    # 无命中：仍注入护栏，降低幻觉政策/SLA
    if not hits:
        return (
            "APPROVED MACROS:\n"
            "No matching macros found.\n"
            "Instruction: Ask for missing details and respond professionally without inventing policies."
        )

    # 有命中：逐条拼 id/intent/title/content
    lines = ["APPROVED MACROS (use and adapt as appropriate):"]
    for h in hits:
        lines.append(
            f"\n[Macro id={h['id']} | intent={h['intent']} | title={h['title']}]\n"
            f"{h['content']}"
        )
    lines.append("\nInstruction: Prefer using these macros; do not invent policy details not present above.")
    return "\n".join(lines)


In [ ]:
# ========== 8) stream_answer：流式 Chat Completions，供 UI 增量刷新 ==========

def stream_answer(client: OpenAI, model: str, messages: List[Dict[str, str]]):
    """创建 stream=True 的补全，边收边 yield 累计文本。"""
    # stream=True：服务端推增量；temperature 用全局固定值
    stream = client.chat.completions.create(
        model=model,                 # Model name
        messages=messages,           # Chat history
        stream=True,                 # Enable streaming
        temperature=TEMPERATURE_FIXED
    )

    # 累计已生成文本
    text = ""

    # 逐 chunk 取 delta.content
    for chunk in stream:
        delta = chunk.choices[0].delta  # Incremental delta
        if delta and delta.content:     # If text content exists
            text += delta.content       # Append to full text
            yield text                  # Yield partial output for UI

    # 再 yield 一次终稿，方便调用方统一收尾
    yield text


In [ ]:
# ========== 9) Judge + Tool-calling（Auto 模式让远程决定是否查 DB） ==========

def judge_two_answers(
    client: OpenAI,
    judge_model: str,
    customer_message: str,
    answer_a: str,
    answer_b: str,
    model_a_name: str,
    model_b_name: str
) -> Dict[str, Any]:
    """让 Judge 比较两条客服回复，返回严格 JSON：分数 / 赢家 / 理由。"""
    # Judge 的 system：评分维度 + 只许 JSON（英文指令保留）
    judge_system_prompt = (
        "You are an impartial judge evaluating two customer-support answers.\n"
        "Score each answer from 0 to 10 based on:\n"
        "1) Factual correctness (no invented policies, SLAs, ETAs)\n"
        "2) Professional tone\n"
        "3) Clarity and actionable next steps\n"
        "4) Completeness given the customer message\n"
        "Return ONLY valid JSON."
    )

    # Judge 的 user：塞入顾客原话与 A/B 两答
    judge_user_prompt = f"""
Customer message:
{customer_message}

Answer A (model: {model_a_name}):
{answer_a}

Answer B (model: {model_b_name}):
{answer_b}

Respond with JSON EXACTLY in this schema:
{{
  "model_A": "{model_a_name}",
  "model_B": "{model_b_name}",
  "score_A": <number 0-10>,
  "score_B": <number 0-10>,
  "winner": "A" or "B" or "tie",
  "reason": "brief concrete explanation citing criteria"
}}
""".strip()

    # response_format=json_object：要求返回可 parse 的 JSON 对象
    resp = client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": judge_system_prompt},
            {"role": "user", "content": judge_user_prompt}
        ],
        response_format={"type": "json_object"}  # Request JSON object
    )

    # 取出 JSON 文本
    verdict_text = resp.choices[0].message.content

    # 解析为 dict
    verdict = json.loads(verdict_text)

    # 最小字段校验：缺键就抛错，避免 UI 静默坏掉
    required = ["model_A", "model_B", "score_A", "score_B", "winner", "reason"]
    for k in required:
        if k not in verdict:
            raise ValueError(f"Judge verdict missing field: {k}")

    if verdict["winner"] not in ["A", "B", "tie"]:
        raise ValueError("Judge winner must be 'A', 'B', or 'tie'")

    return verdict


# ---------- Tools：给远程模型挂 search_macros ----------

def build_search_macros_tool_schema() -> Dict[str, Any]:
    """OpenAI tools schema：允许 REMOTE 模型主动请求查宏库。"""
    return {
        "type": "function",
        "function": {
            "name": "search_macros",
            "description": "Search approved customer support macros from the internal SQLite database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The customer message or a short query to find relevant macros."
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "How many macro hits to return.",
                        "default": 3
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }

SEARCH_MACROS_TOOL = build_search_macros_tool_schema()
TOOLS = [SEARCH_MACROS_TOOL]


def handle_tool_calls_for_macros(message) -> Dict[str, Any]:
    """执行模型发起的 search_macros tool_calls，返回 tool messages + 最后一次 db_result。"""
    tool_messages = []
    last_db_result: Dict[str, Any] = {"query": "", "hits": []}

    for tool_call in (message.tool_calls or []):
        if tool_call.function.name == "search_macros":
            # 解析 arguments JSON（缺省用空对象）
            args = json.loads(tool_call.function.arguments or "{}")
            q = args.get("query", "")
            top_k = int(args.get("top_k", 3))

            # 真正查 SQLite
            last_db_result = search_macros(q, top_k=top_k)

            # tool 角色消息：content 必须是字符串；这里用 JSON
            tool_messages.append({
                "role": "tool",
                "content": json.dumps(last_db_result, ensure_ascii=False),
                "tool_call_id": tool_call.id
            })

    return {"tool_messages": tool_messages, "db_result": last_db_result}


def auto_decide_db_by_remote(
    customer_message: str,
    system_prompt: str,
    cloud_model: str,
    top_k: int = 3,
    max_tool_rounds: int = 3
) -> Dict[str, Any]:
    """Auto 模式：只让远程模型决定是否 tool-call 查库；必要时多轮直到不再要工具。"""
    used_db = False
    db_result: Dict[str, Any] = {"query": "", "hits": []}

    # 决策对话：先不注入宏，逼模型自己决定要不要 search_macros
    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": customer_message.strip()}
    ]

    # tools=TOOLS：打开函数调用
    resp = client_cloud.chat.completions.create(
        model=cloud_model,
        messages=messages,
        tools=TOOLS,
        temperature=TEMPERATURE_FIXED
    )

    rounds = 0
    while resp.choices[0].finish_reason == "tool_calls" and rounds < max_tool_rounds:
        rounds += 1
        used_db = True

        assistant_msg = resp.choices[0].message
        messages.append(assistant_msg)

        handled = handle_tool_calls_for_macros(assistant_msg)
        tool_messages = handled["tool_messages"]
        db_result = handled["db_result"]

        # 把 tool 结果续回对话
        messages.extend(tool_messages)

        # 再问远程：可能继续要工具，或结束
        resp = client_cloud.chat.completions.create(
            model=cloud_model,
            messages=messages,
            tools=TOOLS,
            temperature=TEMPERATURE_FIXED
        )

    # 查过库但 0 hit：仍注入「No matching macros」护栏；没查过则空块
    if used_db:
        # If model asked the tool but db_result is empty, we still provide explicit guidance
        # to avoid hallucinating policies.
        if not db_result.get("hits"):
            approved_block = (
                "APPROVED MACROS:\n"
                f"Query: {db_result.get('query','')}\n"
                "No matching macros found.\n"
                "Instruction: Ask for the minimum necessary details and do not invent policies, SLAs, ETAs, or refunds."
            )
        else:
            approved_block = build_approved_macros_block(db_result)
    else:
        approved_block = ""

    return {"used_db": used_db, "db_result": db_result, "approved_macros_block": approved_block}


In [ ]:
# ========== 10) 可选 TTS：把赢家回复合成 mp3 ==========

def tts_to_file(client: OpenAI, text: str, filename: str = "winner_tts.mp3") -> str:
    """调用 audio.speech，把文本写成 mp3，返回路径给 Gradio Audio。"""
    # TTS 端点：模型名与 voice 保持原设置
    speech = client.audio.speech.create(
        model="gpt-4o-mini-tts",  # TTS model
        voice="onyx",             # Voice name
        input=text                # Text to synthesize
    )

    # 二进制写入磁盘
    with open(filename, "wb") as f:
        f.write(speech.content)

    # 返回路径供 UI 加载
    return filename


In [ ]:
# ========== 11) compare_mode_run：DB 模式 + 双模型流式 + Judge + TTS ==========

def compare_mode_run(
    customer_message: str,
    system_prompt: str,
    cloud_model: str,
    local_model: str,
    judge_model: str,
    db_mode: str,        # "Always" | "Auto" | "Off"
    enable_tts: bool
):
    """
    对比运行器（生成器）：按 DB 模式注入宏 → 流式跑云端/本地 → Judge → 可选 TTS。
    - Off: 从不查库
    - Always: 调用 LLM 前确定性查库
    - Auto: 仅远程模型用 tool-calling 决定是否查库；查了但 0 hit 仍注入护栏
    """

    # 本地后端不可达：提前 yield 警告面板并 return
    if not ollama_ok:
        m1_panel = f"## Model 1 (REMOTE / Cloud) — `{cloud_model}`\n\n⚠️ Local backend unavailable (Ollama not reachable)."
        m2_panel = f"## Model 2 (LOCAL / Ollama) — `{local_model}`\n\n⚠️ Start Ollama with: `ollama serve`."
        j_panel  = f"## Judge (REMOTE) — `{judge_model}`\n\n⚠️ Cannot judge without Model 2."
        yield m1_panel, m2_panel, j_panel, None, "No DB lookup."
        return

    # -------------------------
    # 1) 按 db_mode 决定是否查库 / 如何注入宏
    # -------------------------
    db_trace_md = "No DB lookup."
    approved_macros_block = ""

    mode = (db_mode or "Always").strip()

    if mode == "Off":
        # 完全不查库
        db_trace_md = "DB mode: **Off** (no lookup)."
        approved_macros_block = ""

    elif mode == "Always":
        # 确定性查库，再格式化给 UI
        db_result = search_macros(customer_message, top_k=3)
        db_trace_md = "DB mode: **Always**\n\n" + format_db_trace(db_result)

        # 无论是否命中都 build 块（无命中带护栏）
        approved_macros_block = build_approved_macros_block(db_result)

    elif mode == "Auto":
        # 远程模型决定是否 tool-call
        auto = auto_decide_db_by_remote(
            customer_message=customer_message,
            system_prompt=system_prompt,
            cloud_model=cloud_model,
            top_k=3
        )

        used_db = auto["used_db"]
        db_result = auto["db_result"]
        approved_macros_block = auto["approved_macros_block"]

        if used_db:
            db_trace_md = (
                "DB mode: **Auto**\n"
                "- Remote requested DB: **YES**\n\n"
                + format_db_trace(db_result)
            )
        else:
            db_trace_md = (
                "DB mode: **Auto**\n"
                "- Remote requested DB: **NO**"
            )


    else:
        # 未知模式：退回 Always
        db_result = search_macros(customer_message, top_k=3)
        db_trace_md = "DB mode: **Always** (fallback)\n\n" + format_db_trace(db_result)
        approved_macros_block = build_approved_macros_block(db_result)

    # -------------------------
    # 2) 组装双方共用的 messages
    # -------------------------
    messages: List[Dict[str, str]] = [
        {"role": "system", "content": system_prompt.strip()},
    ]

    # 有宏块才追加第二条 system（权威参考）
    if approved_macros_block:
        messages.append({"role": "system", "content": approved_macros_block})

    messages.append({"role": "user", "content": customer_message.strip()})

    # -------------------------
    # 3) 流式跑 Model 1（云端）
    # -------------------------
    cloud_text = ""
    for partial in stream_answer(client_cloud, cloud_model, messages):
        cloud_text = partial
        m1_panel = f"## Model 1 (REMOTE / Cloud) — `{cloud_model}`\n\n{cloud_text}"
        m2_panel = f"## Model 2 (LOCAL / Ollama) — `{local_model}`\n\n*(waiting...)*"
        j_panel  = f"## Judge (REMOTE) — `{judge_model}`\n\n*(waiting...)*"
        yield m1_panel, m2_panel, j_panel, None, db_trace_md

    # -------------------------
    # 4) 流式跑 Model 2（本地 Ollama）
    # -------------------------
    local_text = ""
    for partial in stream_answer(client_local, local_model, messages):
        local_text = partial
        m1_panel = f"## Model 1 (REMOTE / Cloud) — `{cloud_model}`\n\n{cloud_text}"
        m2_panel = f"## Model 2 (LOCAL / Ollama) — `{local_model}`\n\n{local_text}"
        j_panel  = f"## Judge (REMOTE) — `{judge_model}`\n\n*(waiting...)*"
        yield m1_panel, m2_panel, j_panel, None, db_trace_md

    # -------------------------
    # 5) Judge 打分并挑赢家
    # -------------------------
    verdict = judge_two_answers(
        client=client_cloud,
        judge_model=judge_model,
        customer_message=customer_message.strip(),
        answer_a=cloud_text,
        answer_b=local_text,
        model_a_name=cloud_model,
        model_b_name=local_model
    )

    j_panel = (
        f"## Judge (REMOTE) — `{judge_model}`\n\n"
        f"- Model A (Model 1): **{verdict['model_A']}** — score **{verdict['score_A']}/10**\n"
        f"- Model B (Model 2): **{verdict['model_B']}** — score **{verdict['score_B']}/10**\n"
        f"- Winner: **{verdict['winner']}**\n\n"
        f"**Reason:** {verdict['reason']}"
    )

    # -------------------------
    # 6) 可选 TTS：A 或 tie 用云端答，否则用本地答
    # -------------------------
    winner_text = cloud_text if verdict["winner"] in ["A", "tie"] else local_text
    audio_path = tts_to_file(client_cloud, winner_text) if enable_tts else None

    m1_panel = f"## Model 1 (REMOTE / Cloud) — `{cloud_model}`\n\n{cloud_text}"
    m2_panel = f"## Model 2 (LOCAL / Ollama) — `{local_model}`\n\n{local_text}"

    yield m1_panel, m2_panel, j_panel, audio_path, db_trace_md


In [ ]:
# ========== 12) Gradio UI：输入 / 模型 / DB 模式 / 输出面板 ==========

with gr.Blocks(title="Support Reply Copilot (Compare + Judge + DB + Audio)") as demo:
    # 标题与一句话说明（UI 英文文案保持原样，便于对照课程视频）
    gr.Markdown("# Support Reply Copilot")
    gr.Markdown("Compare two models, judge their replies, optionally use DB macros, and optionally generate TTS for the winner.")

    with gr.Row():
        with gr.Column(scale=1):
            # 顾客原话输入框
            customer_in = gr.Textbox(
                label="Customer message",
                lines=6,
                placeholder=""
            )

            # 一键填入示例工单
            gr.Examples(
                examples=[
                    "Hi, I was charged twice for my subscription this month. Can you fix this and confirm when I’ll get the refund?",
                    "I can’t log in after resetting my password—2FA codes never arrive. Please help ASAP.",
                    "Your service has been down for 30 minutes. What’s the ETA and will we receive a credit?"
                ],
                inputs=[customer_in],
                label="Examples (click to fill)"
            )

            # Model 1：云端下拉
            model1 = gr.Dropdown(
                choices=["gpt-4.1-nano", "gpt-4.1-mini"],
                value=MODEL_1_DEFAULT,
                label="Model 1 (REMOTE / Cloud)"
            )

            # Model 2：本地 Ollama 下拉
            model2 = gr.Dropdown(
                choices=["llama3.1:8b"],
                value=MODEL_2_DEFAULT,
                label="Model 2 (LOCAL / Ollama)"
            )

            # Judge 模型下拉
            judge_model = gr.Dropdown(
                choices=["gpt-4.1-mini", "gpt-4.1-nano"],
                value=JUDGE_DEFAULT,
                label="Judge (REMOTE)"
            )

            gr.Markdown("### Settings")

            # 可编辑 system prompt（默认用上格常量）
            system_prompt_in = gr.Textbox(
                label="System prompt",
                value=DEFAULT_SYSTEM_PROMPT,
                lines=8
            )

            # Always / Auto / Off
            db_mode = gr.Radio(
                choices=["Always", "Auto", "Off"],
                value="Always",
                label="Macro DB usage"
            )

            gr.Markdown(
                "**Always**: The app queries the SQLite macro DB *before* calling any model.\n"
                "**Auto**: The *remote (cloud) model* decides whether to query the DB (tool-calling).\n"
                "**Off**: The DB is never queried."
            )


            # 是否给赢家做 TTS
            enable_tts = gr.Checkbox(value=False, label="Enable TTS (winner audio)")

            run_btn = gr.Button("Run")
            clear_btn = gr.Button("Clear")

        with gr.Column(scale=2):
            # 右侧：两模型回答、Judge、音频、DB 轨迹
            out_m1 = gr.Markdown()
            out_m2 = gr.Markdown()
            out_judge = gr.Markdown()
            out_audio = gr.Audio(label="Winner audio (TTS)", autoplay=False)
            out_db = gr.Markdown("No DB lookup.")

    def clear_all():
        # 清空五个输出槽
        return "", "", "", None, "No DB lookup."

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[out_m1, out_m2, out_judge, out_audio, out_db]
    )

    def run_app(customer_msg, m1, m2, j, sys_prompt, mode, use_tts):
        # 空输入：友好提示，不打 API
        if not customer_msg or not customer_msg.strip():
            yield (
                "## Model 1\n\n⚠️ Please paste a customer message.",
                "## Model 2\n\n*(waiting...)*",
                "## Judge\n\n*(waiting...)*",
                None,
                "No DB lookup."
            )
            return

        # 把 UI 参数转给 compare_mode_run 生成器
        gen = compare_mode_run(
            customer_message=customer_msg,
            system_prompt=sys_prompt,
            cloud_model=m1,
            local_model=m2,
            judge_model=j,
            db_mode=mode,
            enable_tts=use_tts
        )

        # 透传每一次 yield，实现流式刷新
        for a, b, c, audio_path, db_md in gen:
            yield a, b, c, audio_path, db_md

    run_btn.click(
        fn=run_app,
        inputs=[customer_in, model1, model2, judge_model, system_prompt_in, db_mode, enable_tts],
        outputs=[out_m1, out_m2, out_judge, out_audio, out_db]
    )


In [ ]:
# ========== 启动 Gradio：浏览器里打开对比 UI ==========

demo.launch()
